# A4 vertical-slice verification walkthrough

This notebook reproduces the A4 regression checks and a single synthetic patient-to-receptionist flow. Run from the repository root. No real patient data is used.

In [1]:
import platform, sys
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())

Python: 3.13.5
Platform: Linux-6.18.57-x86_64-with-glibc2.41


## 1. Run the complete test suite

The suite includes 36 A2 pure-domain tests and 24 A4 HTTP/API integration tests.

In [2]:
import subprocess
run = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], text=True, capture_output=True)
print(run.stdout)
print(run.stderr)
assert run.returncode == 0, 'Automated test suite failed'


test_cancelled_rejected (test_clinic_model.TestCheckInDecisionTable.test_cancelled_rejected) ... ok
test_completed_is_closed (test_clinic_model.TestCheckInDecisionTable.test_completed_is_closed) ... ok
test_grace_threshold_makes_no_show_eligible_not_automatic (test_clinic_model.TestCheckInDecisionTable.test_grace_threshold_makes_no_show_eligible_not_automatic) ... ok
test_inside_window_accepted (test_clinic_model.TestCheckInDecisionTable.test_inside_window_accepted) ... ok
test_naive_current_time_raises (test_clinic_model.TestCheckInDecisionTable.test_naive_current_time_raises) ... ok
test_negative_grace_raises (test_clinic_model.TestCheckInDecisionTable.test_negative_grace_raises) ... ok
test_pre_grace_end_routes_to_late_review (test_clinic_model.TestCheckInDecisionTable.test_pre_grace_end_routes_to_late_review) ... ok
test_repeat_checkin_is_idempotent_decision (test_clinic_model.TestCheckInDecisionTable.test_repeat_checkin_is_idempotent_decision) ... ok
test_start_boundary_accepted 

## 2. Walk through one end-to-end synthetic example

The clock is fixed. A patient books a window, checks in inside the window, and an authorised receptionist calls the ticket with the current version.

In [3]:
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from fastapi.testclient import TestClient
from app.main import create_app

clock = [datetime(2026, 11, 10, 8, 30, tzinfo=timezone.utc)]
with TemporaryDirectory() as tmp:
    app = create_app(str(Path(tmp) / 'walkthrough.sqlite3'), clock=lambda: clock[0])
    client = TestClient(app)
    patient = client.post('/api/login', json={'username':'patient','password':'PatientDemo!2026'}).json()['access_token']
    headers = {'Authorization': f'Bearer {patient}'}
    booking = client.post('/api/appointments', headers=headers, json={
        'clinic_code':'CLINIC-A',
        'window_start':'2026-11-10T09:00:00+00:00',
        'window_end':'2026-11-10T11:00:00+00:00'
    })
    assert booking.status_code == 201, booking.text
    appointment_id = booking.json()['id']
    clock[0] = datetime(2026, 11, 10, 9, 30, tzinfo=timezone.utc)
    checked_in = client.post(f'/api/appointments/{appointment_id}/check-in', headers={**headers, 'Idempotency-Key':'notebook-key-01'}, json={})
    assert checked_in.status_code == 200, checked_in.text
    ticket = checked_in.json()['ticket']
    receptionist = client.post('/api/login', json={'username':'reception','password':'ReceptionDemo!2026'}).json()['access_token']
    staff_headers = {'Authorization': f'Bearer {receptionist}'}
    queue = client.get('/api/staff/queue?clinic_code=CLINIC-A', headers=staff_headers)
    assert queue.status_code == 200 and len(queue.json()['tickets']) == 1
    called = client.post(f"/api/staff/tickets/{ticket['id']}/call", headers={**staff_headers, 'If-Match':str(ticket['version'])}, json={})
    assert called.status_code == 200, called.text
    print({'appointment_id': appointment_id, 'appointment_state': checked_in.json()['appointment']['state'],
           'ticket_number': ticket['queue_number'], 'ticket_before_call': ticket['state'],
           'ticket_after_call': called.json()['state'], 'version_after_call': called.json()['version']})
    client.close()

{'appointment_id': 1, 'appointment_state': 'CHECKED_IN', 'ticket_number': 'Q001', 'ticket_before_call': 'WAITING', 'ticket_after_call': 'CALLED', 'version_after_call': 2}


## Interpretation and limits

A successful run demonstrates that the prototype's documented state transitions work for this synthetic path. It does not establish real-clinic usability, patient outcome improvements, production security or clinical suitability.